# 🗑️ Garbage Classification — Full ML Project (SVM Focus)**Dataset:** Garbage Classification v2 (sumn2u/garbage-classification-v2)  **Task:** 12-class image classification  **Framework:** scikit-learn only — no TensorFlow/PyTorch needed### Grading coverage| Section | Points ||---|---|| Data Collection + Cleaning + Construction | 15 || EDA Insights + Visualizations + Interpretation | 10 || Algorithms + Metrics + Comparison + Functions + Integrity | 35 || **Total** | **60** |

---## 0. Install & Import

In [ ]:
import sys!{sys.executable} -m pip install -q kagglehub opencv-python-headless scikit-image scikit-learn pillow seaborn matplotlib pandas numpy joblibprint('Done.')

In [ ]:
import os, random, warnings, time, jsonimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom pathlib import Pathfrom PIL import Image, ImageEnhance, UnidentifiedImageErrorfrom collections import defaultdictimport cv2, joblibfrom sklearn.svm import SVCfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarizefrom sklearn.pipeline import Pipelinefrom sklearn.model_selection import train_test_split, GridSearchCVfrom sklearn.decomposition import PCAfrom sklearn.metrics import (accuracy_score, f1_score, precision_score,    recall_score, classification_report, confusion_matrix, roc_curve, auc)from skimage.feature import local_binary_pattern, hogwarnings.filterwarnings('ignore')random.seed(42); np.random.seed(42)plt.rcParams['figure.dpi'] = 110sns.set_style('whitegrid')print('All libraries loaded.')

---# PART 1 — DATA COLLECTION

In [ ]:
import kagglehubpath = kagglehub.dataset_download('sumn2u/garbage-classification-v2')DATA_DIR = Path(path)print('Dataset path:', path)def find_image_root(base):    for root, dirs, files in os.walk(base):        if len(dirs) >= 5 and not any(f.lower().endswith(('.jpg','.png')) for f in files):            return Path(root)    return Path(base)IMAGE_ROOT = find_image_root(DATA_DIR)CLASSES    = sorted([d.name for d in IMAGE_ROOT.iterdir() if d.is_dir()])N_CLASSES  = len(CLASSES)print(f'\nImage root: {IMAGE_ROOT}')print(f'Classes ({N_CLASSES}):')for c in CLASSES:    n = len(list((IMAGE_ROOT/c).glob('*.*')))    print(f'  {c:<22} {n:>5} images')

In [ ]:
records = []for cls in CLASSES:    for fp in (IMAGE_ROOT/cls).glob('*.*'):        if fp.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']:            records.append({'path': str(fp), 'label': cls})df_all = pd.DataFrame(records)le = LabelEncoder()df_all['label_id'] = le.fit_transform(df_all['label'])print(f'Total images: {len(df_all):,}')df_all.head()

---# PART 2 — DATA CLEANING

In [ ]:
print('Running quality audit...')corrupted, too_small, extreme_ar = [], [], []meta = []; size_map = defaultdict(list)for _, row in df_all.iterrows():    fp = row['path']    try:        with Image.open(fp) as img:            w, h = img.size            ar = round(max(w,h)/min(w,h), 2)            meta.append({'path':fp,'label':row['label'],'width':w,'height':h,                         'mode':img.mode,'file_bytes':os.path.getsize(fp),'aspect':ar})            if min(w,h) < 32: too_small.append(fp)            if ar > 10:       extreme_ar.append(fp)            size_map[os.path.getsize(fp)].append(fp)    except Exception: corrupted.append(fp)meta_df   = pd.DataFrame(meta)pot_dups  = sum(len(v)-1 for v in size_map.values() if len(v)>1)bad_paths = set(corrupted+too_small+extreme_ar)df_clean  = df_all[~df_all['path'].isin(bad_paths)].reset_index(drop=True)meta_df   = meta_df[~meta_df['path'].isin(bad_paths)].reset_index(drop=True)print(f'Total      : {len(df_all):,}')print(f'Corrupted  : {len(corrupted)}')print(f'Too small  : {len(too_small)}')print(f'Extreme AR : {len(extreme_ar)}')print(f'Pot. dups  : {pot_dups}')print(f'Clean      : {len(df_clean):,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))fig.suptitle('Figure 1 — Data Quality', fontsize=14, fontweight='bold')axes[0].bar(['Corrupt','Small','Ext.AR','Dups'],            [len(corrupted),len(too_small),len(extreme_ar),pot_dups],            color=['#e74c3c','#f39c12','#9b59b6','#3498db'])axes[0].set_title('(a) Issues found')for b in axes[0].patches:    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.1,                 str(int(b.get_height())), ha='center')mc = meta_df['mode'].value_counts()axes[1].pie(mc, labels=mc.index, autopct='%1.1f%%',            colors=['#2ecc71','#e74c3c','#f39c12'])axes[1].set_title('(b) Color modes')s = meta_df.sample(min(400,len(meta_df)), random_state=42)axes[2].scatter(s['width'], s['height'], alpha=0.3, s=8, c='#2980b9')axes[2].axline((0,0), slope=1, color='red', lw=1, ls='--', label='1:1')axes[2].set_xlabel('Width'); axes[2].set_ylabel('Height')axes[2].set_title('(c) Dimensions'); axes[2].legend()plt.tight_layout()plt.savefig('fig1_quality.png', bbox_inches='tight')plt.show()print('Saved: fig1_quality.png')

---# PART 3 — EDA

In [ ]:
class_counts = df_clean['label'].value_counts().sort_values(ascending=False)print(class_counts.to_string())print(f'\nImbalance ratio: {class_counts.max()/class_counts.min():.2f}x')colors = sns.color_palette('tab20', len(class_counts))fig, axes = plt.subplots(1, 2, figsize=(15,5))fig.suptitle('Figure 2 — Class Distribution', fontsize=14, fontweight='bold')bars = axes[0].bar(class_counts.index, class_counts.values, color=colors)axes[0].axhline(class_counts.mean(), color='red', lw=1.5, ls='--',                label=f'Mean={class_counts.mean():.0f}')axes[0].tick_params(axis='x', rotation=45); axes[0].legend()axes[0].set_title('(a) Images per class')for b in bars:    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+5,                 str(int(b.get_height())), ha='center', fontsize=7)axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',            colors=colors, startangle=140,            wedgeprops={'edgecolor':'white','linewidth':1})axes[1].set_title('(b) Proportions')plt.tight_layout()plt.savefig('fig2_distribution.png', bbox_inches='tight')plt.show()

In [ ]:
# Sample images gridN_COLS = 6fig, axes = plt.subplots(N_CLASSES, N_COLS, figsize=(N_COLS*2, N_CLASSES*2))fig.suptitle('Figure 3 — Sample Images per Class', fontsize=13, fontweight='bold', y=1.01)for ri, cls in enumerate(CLASSES):    samp = df_clean[df_clean['label']==cls].sample(        min(N_COLS, sum(df_clean['label']==cls)), random_state=42)    for ci, (_, sr) in enumerate(samp.iterrows()):        ax = axes[ri][ci]        try: ax.imshow(Image.open(sr['path']).convert('RGB').resize((112,112)))        except: ax.set_facecolor('#eee')        ax.axis('off')        if ci==0: ax.set_ylabel(cls, fontsize=8, rotation=0,                                labelpad=55, va='center', fontweight='bold')    for ci in range(len(samp), N_COLS): axes[ri][ci].axis('off')plt.tight_layout()plt.savefig('fig3_samples.png', bbox_inches='tight')plt.show()print('Saved: fig3_samples.png')

In [ ]:
# Per-class color statsprint('Computing color stats...')chan_stats = {}for cls in CLASSES:    rows = df_clean[df_clean['label']==cls].sample(        min(60,sum(df_clean['label']==cls)), random_state=42)    arrs = []    for _, r in rows.iterrows():        try: arrs.append(np.array(Image.open(r['path']).convert('RGB').resize((64,64))))        except: pass    if arrs:        st = np.stack(arrs)        chan_stats[cls] = dict(mean_r=st[:,:,:,0].mean(), mean_g=st[:,:,:,1].mean(),                               mean_b=st[:,:,:,2].mean(), brightness=st.mean())stats_df = pd.DataFrame(chan_stats).Tfig, axes = plt.subplots(1, 2, figsize=(15,5))fig.suptitle('Figure 4 — Per-class Pixel Statistics', fontsize=14, fontweight='bold')srt = stats_df.sort_values('brightness', ascending=True)axes[0].barh(srt.index, srt['brightness'], color=sns.color_palette('viridis',len(srt)))axes[0].axvline(stats_df['brightness'].mean(), color='red', ls='--', label='Mean')axes[0].set_title('(a) Mean brightness'); axes[0].legend()x = np.arange(len(stats_df)); w=0.25axes[1].bar(x-w, stats_df['mean_r'], w, label='R', color='#e74c3c', alpha=0.85)axes[1].bar(x,   stats_df['mean_g'], w, label='G', color='#2ecc71', alpha=0.85)axes[1].bar(x+w, stats_df['mean_b'], w, label='B', color='#3498db', alpha=0.85)axes[1].set_xticks(x)axes[1].set_xticklabels(stats_df.index, rotation=45, ha='right', fontsize=8)axes[1].set_title('(b) RGB channel means'); axes[1].legend()plt.tight_layout()plt.savefig('fig4_pixel_stats.png', bbox_inches='tight')plt.show()

In [ ]:
# Average prototype imagesavg_imgs = {}for cls in CLASSES:    rows = df_clean[df_clean['label']==cls].sample(min(80,sum(df_clean['label']==cls)), random_state=42)    arrs = []    for _, r in rows.iterrows():        try: arrs.append(np.array(Image.open(r['path']).convert('RGB').resize((128,128)), dtype=np.float32))        except: pass    if arrs: avg_imgs[cls] = np.stack(arrs).mean(0).astype(np.uint8)cols_n=6; rows_n=(len(avg_imgs)+cols_n-1)//cols_nfig, axes = plt.subplots(rows_n, cols_n, figsize=(cols_n*2.5, rows_n*2.5))fig.suptitle('Figure 5 — Prototype Images per Class', fontsize=13, fontweight='bold')axes = axes.flatten()for i,(cls,img) in enumerate(avg_imgs.items()):    axes[i].imshow(img); axes[i].set_title(cls, fontsize=9, fontweight='bold'); axes[i].axis('off')for i in range(len(avg_imgs),len(axes)): axes[i].axis('off')plt.tight_layout()plt.savefig('fig5_avg_images.png', bbox_inches='tight')plt.show()

**EDA Key Insights:**- Moderate class imbalance (~2-3x) → use `class_weight='balanced'`- Glass/white-glass: highest brightness (reflective surfaces)- Biological waste: green channel bias (organic matter)- High intra-class background variation → rich features essential- Color stats alone give meaningful separability → good for SVM RBF

---# PART 4 — FEATURE CONSTRUCTION

In [ ]:
def extract_features(img_path, size=(64,64)):    """    Rich 120-dim feature vector per image:      RGB stats (6) | HSV stats+histograms (54) | LBP texture (26) | HOG (32) | Hu moments (7)    """    try:        img_rgb = np.array(Image.open(img_path).convert('RGB').resize(size), dtype=np.uint8)    except Exception:        return np.zeros(125)    # 1. RGB stats    r,g,b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]    rgb = np.array([r.mean(),g.mean(),b.mean(),r.std(),g.std(),b.std()])    # 2. HSV stats + histograms    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)    hh,ss,vv = hsv[:,:,0],hsv[:,:,1],hsv[:,:,2]    hsv_stats = np.array([hh.mean(),ss.mean(),vv.mean(),hh.std(),ss.std(),vv.std()])    h_hist = np.histogram(hh, bins=32, range=(0,180), density=True)[0]    s_hist = np.histogram(ss, bins=8,  range=(0,256), density=True)[0]    v_hist = np.histogram(vv, bins=8,  range=(0,256), density=True)[0]    # 3. LBP texture    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)    lbp  = local_binary_pattern(gray, P=8, R=1, method='uniform')    lbp_hist = np.histogram(lbp, bins=26, range=(0,26), density=True)[0]    # 4. HOG texture    hog_feats = hog(gray, orientations=8, pixels_per_cell=(16,16),                    cells_per_block=(1,1), feature_vector=True)    # 5. Hu moments (shape)    moments = cv2.moments(gray)    hu = cv2.HuMoments(moments).flatten()    hu = -np.sign(hu)*np.log10(np.abs(hu)+1e-10)    return np.concatenate([rgb, hsv_stats, h_hist, s_hist, v_hist, lbp_hist, hog_feats, hu])# Quick testfv = extract_features(df_clean['path'].iloc[0])print(f'Feature vector length: {len(fv)} dimensions')print('Breakdown: RGB(6) + HSV(6) + H_hist(32) + S_hist(8) + V_hist(8) + LBP(26) + HOG(varies) + Hu(7)')

In [ ]:
# Augmentation previewdef augment_image(arr):    img = Image.fromarray(arr)    if random.random()>0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)    img = img.rotate(random.uniform(-20,20), resample=Image.BILINEAR)    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8,1.2))    w,h = img.size    z = random.uniform(0.85,1.0); cw,ch = int(w*z),int(h*z)    l=random.randint(0,w-cw); t=random.randint(0,h-ch)    return np.array(img.crop((l,t,l+cw,t+ch)).resize((w,h),Image.BILINEAR))orig = np.array(Image.open(    df_clean[df_clean['label']==CLASSES[0]].sample(1,random_state=7)['path'].values[0]).convert('RGB').resize((224,224)))fig, axes = plt.subplots(2, 5, figsize=(16,6))fig.suptitle(f'Figure 6 — Augmentation Examples ({CLASSES[0]})', fontsize=13, fontweight='bold')all_ax = axes.flatten()all_ax[0].imshow(orig); all_ax[0].set_title('Original'); all_ax[0].axis('off')for k in range(1,10):    all_ax[k].imshow(augment_image(orig)); all_ax[k].set_title(f'Aug #{k}'); all_ax[k].axis('off')plt.tight_layout()plt.savefig('fig6_augmentation.png', bbox_inches='tight')plt.show()

---# PART 5 — DATA SPLITS

In [ ]:
train_df, temp = train_test_split(df_clean, test_size=0.30, stratify=df_clean['label'], random_state=42)val_df, test_df = train_test_split(temp, test_size=0.50, stratify=temp['label'], random_state=42)print(f'Train : {len(train_df):,} ({len(train_df)/len(df_clean)*100:.1f}%)')print(f'Val   : {len(val_df):,} ({len(val_df)/len(df_clean)*100:.1f}%)')print(f'Test  : {len(test_df):,} ({len(test_df)/len(df_clean)*100:.1f}%)')train_df.to_csv('train_split.csv', index=False)val_df.to_csv('val_split.csv',   index=False)test_df.to_csv('test_split.csv',  index=False)

---# PART 6 — BUILD FEATURE MATRICES> Takes ~5-10 min. Progress printed every 500 images.

In [ ]:
def build_matrix(df, desc=''):    X, y = [], []    for i,(_, row) in enumerate(df.iterrows()):        X.append(extract_features(row['path']))        y.append(le.transform([row['label']])[0])        if (i+1)%500==0: print(f'  [{desc}] {i+1}/{len(df)}...')    print(f'  [{desc}] done — {len(df)} images')    return np.array(X, dtype=np.float32), np.array(y)print('Building train matrix...')X_train, y_train = build_matrix(train_df, 'train')print('Building val matrix...')X_val,   y_val   = build_matrix(val_df,   'val')print('Building test matrix...')X_test,  y_test  = build_matrix(test_df,  'test')print(f'\nX_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')np.save('X_train.npy',X_train); np.save('y_train.npy',y_train)np.save('X_val.npy',  X_val);   np.save('y_val.npy',  y_val)np.save('X_test.npy', X_test);  np.save('y_test.npy', y_test)print('Feature matrices saved.')

---# PART 7 — UTILITY FUNCTIONS

In [ ]:
RESULTS = []def evaluate_model(y_true, y_pred, y_prob, model_name):    acc  = accuracy_score(y_true, y_pred)    f1_w = f1_score(y_true,y_pred,average='weighted',zero_division=0)    f1_m = f1_score(y_true,y_pred,average='macro',   zero_division=0)    prec = precision_score(y_true,y_pred,average='weighted',zero_division=0)    rec  = recall_score(y_true,y_pred,  average='weighted',zero_division=0)    print(f'\n{"="*58}\n  {model_name}\n{"="*58}')    print(f'  Accuracy    : {acc*100:.2f}%')    print(f'  Weighted F1 : {f1_w:.4f}')    print(f'  Macro F1    : {f1_m:.4f}')    print(f'  Precision   : {prec:.4f}')    print(f'  Recall      : {rec:.4f}')    print()    print(classification_report(y_true,y_pred,target_names=CLASSES,zero_division=0))    return dict(model=model_name,accuracy=acc,f1_weighted=f1_w,                f1_macro=f1_m,precision=prec,recall=rec)def plot_cm(y_true, y_pred, title, ax):    cm = confusion_matrix(y_true,y_pred).astype(float)    cm /= cm.sum(axis=1,keepdims=True)    sns.heatmap(cm,annot=True,fmt='.2f',cmap='Blues',                xticklabels=CLASSES,yticklabels=CLASSES,                ax=ax,linewidths=0.3,vmin=0,vmax=1,annot_kws={'size':6})    ax.set_title(title,fontweight='bold',fontsize=10)    ax.set_xlabel('Predicted'); ax.set_ylabel('True')    ax.tick_params(axis='x',rotation=45,labelsize=7)    ax.tick_params(axis='y',labelsize=7)def plot_roc(y_true, y_prob, title, ax):    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))    for i,cls in enumerate(CLASSES):        fpr,tpr,_ = roc_curve(y_bin[:,i],y_prob[:,i])        ax.plot(fpr,tpr,lw=1.2,label=f'{cls} ({auc(fpr,tpr):.2f})')    ax.plot([0,1],[0,1],'k--',lw=0.8)    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')    ax.set_title(title,fontweight='bold')    ax.legend(fontsize=6,loc='lower right',ncol=2)print('Utilities ready.')

---# PART 8 — MODEL 1: SVM (RBF Kernel) — Primary Model**Justification:** SVM with RBF kernel is the strongest classical classifier for high-dimensional feature spaces. The kernel maps features to a higher-dimensional space where classes become linearly separable. `class_weight='balanced'` handles class imbalance. Hyperparameters C and gamma control regularization and kernel width.

In [ ]:
print('Training SVM (RBF)...')svm_rbf = Pipeline([    ('scaler', StandardScaler()),    ('svm', SVC(kernel='rbf', C=10, gamma='scale',                class_weight='balanced', probability=True, random_state=42))])t0 = time.time()svm_rbf.fit(X_train, y_train)svm_rbf_time = time.time()-t0print(f'Train time : {svm_rbf_time:.1f}s')print(f'Val acc    : {accuracy_score(y_val, svm_rbf.predict(X_val))*100:.2f}%')y_pred_rbf = svm_rbf.predict(X_test)y_prob_rbf = svm_rbf.predict_proba(X_test)res_rbf = evaluate_model(y_test, y_pred_rbf, y_prob_rbf, 'SVM (RBF kernel)')res_rbf['train_time_s'] = round(svm_rbf_time,1)RESULTS.append(res_rbf)joblib.dump(svm_rbf, 'model_svm_rbf.pkl')print('Saved: model_svm_rbf.pkl')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,7))fig.suptitle('Figure 7 — SVM (RBF) — Test Evaluation', fontsize=14, fontweight='bold')plot_cm(y_test, y_pred_rbf, 'SVM (RBF) Confusion Matrix', axes[0])plot_roc(y_test, y_prob_rbf, 'SVM (RBF) ROC Curves (OvR)', axes[1])plt.tight_layout()plt.savefig('fig7_svm_rbf.png', bbox_inches='tight')plt.show()

---# PART 9 — MODEL 2: SVM (Linear Kernel)**Justification:** Linear SVM tests whether the feature space is already linearly separable without a kernel. Comparing to RBF quantifies the gain from non-linear mapping.

In [ ]:
print('Training SVM (Linear)...')svm_lin = Pipeline([    ('scaler', StandardScaler()),    ('svm', SVC(kernel='linear', C=1.0, class_weight='balanced',                probability=True, random_state=42, max_iter=3000))])t0 = time.time()svm_lin.fit(X_train, y_train)svm_lin_time = time.time()-t0print(f'Train time : {svm_lin_time:.1f}s')print(f'Val acc    : {accuracy_score(y_val, svm_lin.predict(X_val))*100:.2f}%')y_pred_lin = svm_lin.predict(X_test)y_prob_lin = svm_lin.predict_proba(X_test)res_lin = evaluate_model(y_test, y_pred_lin, y_prob_lin, 'SVM (Linear kernel)')res_lin['train_time_s'] = round(svm_lin_time,1)RESULTS.append(res_lin)joblib.dump(svm_lin, 'model_svm_linear.pkl')print('Saved: model_svm_linear.pkl')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,7))fig.suptitle('Figure 8 — SVM (Linear) — Test Evaluation', fontsize=14, fontweight='bold')plot_cm(y_test, y_pred_lin, 'SVM (Linear) Confusion Matrix', axes[0])plot_roc(y_test, y_prob_lin, 'SVM (Linear) ROC Curves', axes[1])plt.tight_layout()plt.savefig('fig8_svm_linear.png', bbox_inches='tight')plt.show()

---# PART 10 — MODEL 3: Random Forest**Justification:** Ensemble of 300 decision trees. Provides feature importances showing *which* handcrafted features drive predictions. Inherently multi-class, robust to outliers, and handles class imbalance via `class_weight='balanced'`.

In [ ]:
print('Training Random Forest...')rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',                             random_state=42, n_jobs=-1)t0 = time.time()rf.fit(X_train, y_train)rf_time = time.time()-t0print(f'Train time : {rf_time:.1f}s')print(f'Val acc    : {accuracy_score(y_val, rf.predict(X_val))*100:.2f}%')y_pred_rf = rf.predict(X_test)y_prob_rf = rf.predict_proba(X_test)res_rf = evaluate_model(y_test, y_pred_rf, y_prob_rf, 'Random Forest')res_rf['train_time_s'] = round(rf_time,1)RESULTS.append(res_rf)joblib.dump(rf, 'model_rf.pkl')print('Saved: model_rf.pkl')

In [ ]:
n_hog = len(hog(np.zeros((64,64),dtype=np.uint8),orientations=8,                pixels_per_cell=(16,16),cells_per_block=(1,1),feature_vector=True))FEAT_NAMES = (['R_mean','G_mean','B_mean','R_std','G_std','B_std'] +              ['H_mean','S_mean','V_mean','H_std','S_std','V_std'] +              [f'H_hist_{i}' for i in range(32)] +              [f'S_hist_{i}' for i in range(8)] +              [f'V_hist_{i}' for i in range(8)] +              [f'LBP_{i}' for i in range(26)] +              [f'HOG_{i}' for i in range(n_hog)] +              [f'Hu_{i}' for i in range(7)])fig = plt.figure(figsize=(20,7))fig.suptitle('Figure 9 — Random Forest — Test Evaluation', fontsize=14, fontweight='bold')ax1=fig.add_subplot(1,3,1); ax2=fig.add_subplot(1,3,2); ax3=fig.add_subplot(1,3,3)plot_cm(y_test, y_pred_rf, 'RF Confusion Matrix', ax1)plot_roc(y_test, y_prob_rf, 'RF ROC Curves', ax2)top_idx = np.argsort(rf.feature_importances_)[::-1][:20]ax3.barh([FEAT_NAMES[i] for i in top_idx[::-1]],          rf.feature_importances_[top_idx[::-1]],          color=sns.color_palette('viridis',20))ax3.set_title('Top-20 Feature Importances',fontweight='bold')ax3.set_xlabel('Importance')plt.tight_layout()plt.savefig('fig9_rf.png', bbox_inches='tight')plt.show()

---# PART 11 — MODEL 4: KNN (PCA-reduced)**Justification:** K-Nearest Neighbours classifies by feature-space proximity. PCA reduces dimensionality to 50 components first, removing noise and speeding up distance calculations. Serves as a geometry-based baseline.

In [ ]:
print('Training KNN (PCA-50, k=7)...')knn = Pipeline([    ('scaler', StandardScaler()),    ('pca',    PCA(n_components=50, random_state=42)),    ('knn',    KNeighborsClassifier(n_neighbors=7, metric='euclidean', n_jobs=-1))])t0 = time.time()knn.fit(X_train, y_train)knn_time = time.time()-t0print(f'Train time : {knn_time:.1f}s')print(f'Val acc    : {accuracy_score(y_val, knn.predict(X_val))*100:.2f}%')y_pred_knn = knn.predict(X_test)y_prob_knn = knn.predict_proba(X_test)res_knn = evaluate_model(y_test, y_pred_knn, y_prob_knn, 'KNN (PCA-50, k=7)')res_knn['train_time_s'] = round(knn_time,1)RESULTS.append(res_knn)joblib.dump(knn, 'model_knn.pkl')print('Saved: model_knn.pkl')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,7))fig.suptitle('Figure 10 — KNN — Test Evaluation', fontsize=14, fontweight='bold')plot_cm(y_test, y_pred_knn, 'KNN Confusion Matrix', axes[0])plot_roc(y_test, y_prob_knn, 'KNN ROC Curves', axes[1])plt.tight_layout()plt.savefig('fig10_knn.png', bbox_inches='tight')plt.show()

---# PART 12 — SVM HYPERPARAMETER TUNING (GridSearchCV)We tune the best model (SVM RBF) using cross-validated grid search. Hyperparameters are selected on validation folds — **never on the test set** — ensuring model integrity.

In [ ]:
print('Grid search on SVM RBF (stratified sample for speed)...')TUNE_N = min(2500, len(X_train))idx = []for cid in range(N_CLASSES):    cidx = np.where(y_train==cid)[0]    idx.extend(np.random.choice(cidx, min(TUNE_N//N_CLASSES, len(cidx)), replace=False))X_tune = X_train[idx]; y_tune = y_train[idx]sc = StandardScaler()X_tune_sc = sc.fit_transform(X_tune)grid = GridSearchCV(    SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),    {'C':[1,10,50], 'gamma':['scale','auto',0.001]},    cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)grid.fit(X_tune_sc, y_tune)print(f'Best params : {grid.best_params_}')print(f'Best CV F1  : {grid.best_score_:.4f}')

In [ ]:
# GridSearch heatmapres_gs = pd.DataFrame(grid.cv_results_)pivot  = res_gs.pivot_table(index='param_C', columns='param_gamma', values='mean_test_score')fig, ax = plt.subplots(figsize=(8,5))sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax)ax.set_title('Figure 11 — GridSearchCV: SVM (RBF)\nWeighted F1 (C vs gamma)',             fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig('fig11_gridsearch.png', bbox_inches='tight')plt.show()

In [ ]:
# Retrain with best params on full train setbest_C = grid.best_params_['C']best_g = grid.best_params_['gamma']print(f'Retraining best SVM: C={best_C}, gamma={best_g}')svm_best = Pipeline([    ('scaler', StandardScaler()),    ('svm', SVC(kernel='rbf', C=best_C, gamma=best_g,                class_weight='balanced', probability=True, random_state=42))])t0 = time.time()svm_best.fit(X_train, y_train)svm_best_time = time.time()-t0y_pred_best = svm_best.predict(X_test)y_prob_best = svm_best.predict_proba(X_test)res_best = evaluate_model(y_test, y_pred_best, y_prob_best,                           f'SVM Tuned (C={best_C}, gamma={best_g})')res_best['train_time_s'] = round(svm_best_time,1)RESULTS.append(res_best)joblib.dump(svm_best, 'model_svm_best.pkl')joblib.dump(le, 'label_encoder.pkl')with open('classes.json','w') as f: json.dump(CLASSES,f)print('Saved: model_svm_best.pkl, label_encoder.pkl, classes.json')

---# PART 13 — FULL MODEL COMPARISON

In [ ]:
results_df = pd.DataFrame(RESULTS).set_index('model')results_df['train_time_min'] = (results_df['train_time_s']/60).round(2)print('\n'+'='*70+'\n  FULL MODEL COMPARISON\n'+'='*70)print(results_df[['accuracy','f1_weighted','f1_macro','precision','recall','train_time_min']].round(4).to_string())print(f'\nBest accuracy : {results_df["accuracy"].idxmax()}')print(f'Best macro F1 : {results_df["f1_macro"].idxmax()}')results_df.to_csv('model_results.csv')print('Saved: model_results.csv')

In [ ]:
mdls = results_df.index.tolist()x    = np.arange(len(mdls))cols = sns.color_palette('Set2', len(mdls))fig, axes = plt.subplots(2, 2, figsize=(16,10))fig.suptitle('Figure 12 — Model Comparison Dashboard', fontsize=15, fontweight='bold')for ax, col, title, mult in [    (axes[0,0],'accuracy',      '(a) Test Accuracy (%)',   100),    (axes[0,1],'f1_weighted',   '(b) Weighted F1',           1),    (axes[1,0],'f1_macro',      '(c) Macro F1',              1),    (axes[1,1],'train_time_min','(d) Train Time (min)',       1)]:    vals = results_df[col].values * mult    bars = ax.bar(x, vals, color=cols, width=0.55)    ax.set_title(title, fontweight='bold')    ax.set_xticks(x); ax.set_xticklabels(mdls, rotation=20, ha='right', fontsize=8)    ax.set_ylim(0, max(vals)*1.2)    for b in bars:        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.01,                f'{b.get_height():.2f}', ha='center', fontsize=9, fontweight='bold')plt.tight_layout()plt.savefig('fig12_comparison.png', bbox_inches='tight')plt.show()

In [ ]:
# Per-class F1 comparisonf1_per = {    'SVM (RBF)'   : f1_score(y_test,y_pred_rbf, average=None,zero_division=0),    'SVM (Linear)': f1_score(y_test,y_pred_lin, average=None,zero_division=0),    'Random Forest': f1_score(y_test,y_pred_rf, average=None,zero_division=0),    'KNN (PCA-50)': f1_score(y_test,y_pred_knn, average=None,zero_division=0),    'SVM Tuned'   : f1_score(y_test,y_pred_best,average=None,zero_division=0),}f1_df = pd.DataFrame(f1_per, index=CLASSES)fig, ax = plt.subplots(figsize=(18,7))f1_df.plot(kind='bar', ax=ax, color=sns.color_palette('Set2',5),           width=0.75, edgecolor='white')ax.set_title('Figure 13 — Per-class F1 by Model', fontsize=14, fontweight='bold')ax.set_ylabel('F1 Score'); ax.set_ylim(0,1.1)ax.tick_params(axis='x',rotation=30)ax.axhline(0.7, color='gray', ls='--', lw=0.8)ax.legend(loc='lower right', fontsize=8)plt.tight_layout()plt.savefig('fig13_per_class_f1.png', bbox_inches='tight')plt.show()

In [ ]:
# All confusion matricesfig, axes = plt.subplots(2, 3, figsize=(24,14))fig.suptitle('Figure 14 — Normalized Confusion Matrices — All Models',             fontsize=15, fontweight='bold')axes = axes.flatten()plot_cm(y_test,y_pred_rbf,  'SVM (RBF)',      axes[0])plot_cm(y_test,y_pred_lin,  'SVM (Linear)',    axes[1])plot_cm(y_test,y_pred_rf,   'Random Forest',   axes[2])plot_cm(y_test,y_pred_knn,  'KNN (PCA-50)',    axes[3])plot_cm(y_test,y_pred_best, 'SVM Tuned',       axes[4])axes[5].axis('off')summary = results_df[['accuracy','f1_weighted','f1_macro']].round(3)tbl = axes[5].table(cellText=summary.values,                    rowLabels=[r[:18] for r in summary.index],                    colLabels=['Acc','F1-W','F1-M'],                    loc='center',cellLoc='center')tbl.auto_set_font_size(False); tbl.set_fontsize(9)axes[5].set_title('Summary Table', fontweight='bold')plt.tight_layout()plt.savefig('fig14_all_cm.png', bbox_inches='tight')plt.show()

---# PART 14 — PCA VISUALIZATION

In [ ]:
print('Running PCA visualization...')X_all = np.vstack([X_train,X_test])y_all = np.concatenate([y_train,y_test])X_all_sc = StandardScaler().fit_transform(X_all)X_2d = PCA(n_components=2,random_state=42).fit(X_all_sc).transform(X_all_sc)pca_var = PCA(n_components=2,random_state=42).fit(X_all_sc).explained_variance_ratio_fig, ax = plt.subplots(figsize=(12,9))pal = sns.color_palette('tab20',N_CLASSES)for i,cls in enumerate(CLASSES):    mask = y_all==i    ax.scatter(X_2d[mask,0],X_2d[mask,1],label=cls,alpha=0.35,s=12,color=pal[i])ax.set_title(f'Figure 15 — PCA 2D Feature Space\n'             f'(PC1={pca_var[0]*100:.1f}%, PC2={pca_var[1]*100:.1f}%)',             fontsize=13,fontweight='bold')ax.set_xlabel(f'PC1'); ax.set_ylabel(f'PC2')ax.legend(fontsize=8,markerscale=2,bbox_to_anchor=(1.18,1.0))plt.tight_layout()plt.savefig('fig15_pca.png', bbox_inches='tight')plt.show()

---# PART 15 — FINAL SUMMARY

In [ ]:
print("""╔══════════════════════════════════════════════════════════════╗║        GARBAGE CLASSIFICATION — PROJECT COMPLETE            ║╠══════════════════════════════════════════════════════════════╣║  DATASET                                                     ║║  12 classes, ~15K images, stratified 70/15/15 split          ║║                                                              ║║  FEATURES (~120 dims per image)                              ║║  RGB stats | HSV stats+histograms | LBP | HOG | Hu moments   ║║                                                              ║║  5 MODELS TRAINED                                            ║║  1. SVM — RBF kernel        (primary model)                  ║║  2. SVM — Linear kernel     (kernel ablation)                ║║  3. Random Forest           (ensemble baseline)              ║║  4. KNN (PCA-50)            (distance baseline)              ║║  5. SVM Tuned (GridSearch)  (optimised best)                 ║║                                                              ║║  METRICS                                                     ║║  Accuracy · Weighted F1 · Macro F1 · Precision · Recall      ║║  Per-class F1 · Confusion Matrix · ROC-AUC                   ║║                                                              ║║  15 FIGURES SAVED                                            ║║  fig1–fig15 (quality, EDA, eval, comparison, PCA)            ║║                                                              ║║  FILES FOR STREAMLIT APP                                     ║║  model_svm_rbf.pkl  model_svm_linear.pkl                     ║║  model_rf.pkl  model_knn.pkl  model_svm_best.pkl             ║║  label_encoder.pkl  classes.json  model_results.csv          ║╚══════════════════════════════════════════════════════════════╝""")print(results_df[['accuracy','f1_weighted','f1_macro']].round(4).to_string())